# Feature Selection

### Importing and setting up

In [14]:
import pandas as pd
import numpy as np
df_f = pd.read_csv("../../Data/JNPA_feature_engineered.csv")
df_f["Date"] = pd.to_datetime(df_f["Date"])
df_f.head()

,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,Exp_Transit_CFS,Exp_Transit_ICD,Total_TEUs_Handled,Date,Volume_Lag1,Rail_Friction,Is_Monsoon,Stress,Risk
0,Feb 23,27.8,24.0,69.6,2.55,2.78,79.0,69.5,67.8,80.7,5.06,109.2,467614,2023-02-01,522592.0,2.900000,0,0,1
1,Mar 23,26.5,22.3,63.7,3.65,2.47,97.1,74.0,72.4,86.1,6.42,113.0,560076,2023-03-01,467614.0,2.856502,0,0,0
2,Apr 23,29.9,25.4,53.2,4.00,2.62,90.1,80.0,77.1,104.6,6.46,89.3,503668,2023-04-01,560076.0,2.094488,0,0,1
3,May 23,19.9,17.2,51.0,2.33,2.56,91.1,65.0,62.8,81.3,4.21,109.6,538247,2023-05-01,503668.0,2.965116,0,0,0
4,Jun 23,15.8,13.8,38.4,2.18,2.35,107.0,71.9,69.8,88.5,4.18,112.2,523948,2023-06-01,538247.0,2.782609,1,523948,0


## Dataset Input

The dataset used in this notebook is the feature-engineered dataset generated in the previous stage of the analysis pipeline. This dataset contains operational variables derived from the cleaned JNPA logistics dataset, along with engineered features and the congestion risk indicator.

The purpose of this stage is to identify suitable predictors from this dataset before training the congestion prediction model.

## Objective

The purpose of the feature selection stage is to identify a small set of operational predictors that can explain congestion risk within the port system.

The dataset used in this study contains a limited number of monthly observations. Therefore, the number of predictors must be restricted to avoid model overfitting.

This constraint can be expressed using the Events Per Variable (EPV) guideline used in logistic regression:

EPV = number of positive events / number of predictors

To maintain model stability, EPV ≥ 5 is recommended. Given the number of congestion events in the dataset, this limits the model to approximately two or three predictors.

The feature selection process therefore focuses on identifying a small set of operational variables that are both statistically appropriate and operationally meaningful.

## Exclusion of Direct Dwell Variables

The congestion risk label used in this study is derived from the import dwell time variable.

Risk = 1 if Import_Dwell_Overall exceeds the selected congestion threshold.

Because the target variable is constructed directly from import dwell time, including Import_Dwell_Overall as a predictor would cause data leakage. In such a case the model would simply learn the rule used to construct the label rather than discovering operational patterns that lead to congestion.

Therefore variables that directly represent the same dwell signal must be excluded from the predictor set.

In [15]:
df_f[["Imp_Dwell_Overall","Imp_Dwell_Truck"]].corr()

,Imp_Dwell_Overall,Imp_Dwell_Truck
Imp_Dwell_Overall,1.000000,0.990384
Imp_Dwell_Truck,0.990384,1.000000


The exploratory data analysis showed that Import_Dwell_Truck is almost perfectly correlated with Import_Dwell_Overall.

This indicates that both variables represent the same operational signal.

Including either of these variables as predictors would effectively allow the model to directly observe the target variable.

Therefore both Import_Dwell_Overall and Import_Dwell_Truck are excluded from the candidate predictor set.

## Consideration of Export Dwell

Export dwell represents the staging time of containers waiting for vessel loading.

Operationally, export containers are typically delivered to the terminal within the vessel loading window. As a result, export dwell frequently reflects planned staging time rather than congestion conditions.

Because export dwell primarily reflects vessel scheduling rather than yard congestion, it is not used as a primary predictor of congestion risk.

Instead, operational indicators such as parking dwell are used to capture export-side pressure on terminal infrastructure.

## Consideration of ICD Transit Variables

The dataset also includes variables representing container movement between the port and Inland Container Depots (ICDs).

ICD transit variables represent inland rail movement between the port terminal and inland logistics nodes. While delays in ICD movement may affect overall supply chain efficiency, the exploratory data analysis indicated that these variables have weak correlation with import dwell time.

This suggests that ICD transit delays occur largely outside the port terminal and therefore have limited direct influence on terminal yard congestion.

For this reason, ICD-related variables are retained in the dataset for diagnostic interpretation but are not included as primary predictors in the congestion risk model.

## Selected Feature Predictors

After excluding direct dwell variables and evaluating additional operational variables, the remaining predictors are derived from operational conditions observed during the exploratory analysis.

These predictors represent different sources of congestion pressure within the port system.

• Volume_Lag1 – container throughput pressure  
• Rail_Friction – imbalance between rail and truck evacuation performance  
• Stress – interaction between container volume and seasonal conditions  
• Parking_Dwell – gate and yard congestion indicator

These variables represent operational drivers of congestion rather than the congestion outcome itself.

## Operational Feature Groups

Two operational groups of predictors are considered.

Group A represents import evacuation pressure, which focuses on factors affecting the movement of containers out of the port.
### Group A – Import evacuation pressure:Volume_Lag1,Rail_Friction,Stress


Group B represents export yard pressure, which captures congestion generated by export truck arrivals and terminal gate activity.
### Group B – Export yard pressure:Volume_Lag1,Rail_Friction,Parking_Dwell

These groups will be evaluated in the subsequent modelling stage to determine which operational dynamics better explain congestion risk.

In [16]:
# Group A – Import evacuation pressure
features_A = ["Volume_Lag1","Rail_Friction","Stress"]

# Group B – Export yard pressure
features_B = ["Volume_Lag1","Rail_Friction","Parking_Dwell"]